# **PyTorch Implementation of Negative Sampling**

## **The Negative Sampling Architecture**

To implement the Skip-Gram Word2Vec model with **Negative Sampling**, we pivot from the standard "Predict a neighbor word" Softmax approach (which is computationally expensive for large vocabularies) toward a high-speed **Binary Classification** task. Instead of calculating a probability distribution over the entire vocabulary, the model takes two inputs—a **Context word** and a **Target word**—and predicts whether they are true neighbors ($1$) or a random pair ($0$).

### **The Training & Deployment Workflow**

The following steps outline the end-to-end process for training the model and preserving the learned embeddings for downstream tasks:

1.  **Prepare and Load the Dataset:** Tokenize the corpus and generate (context, positive target, negative samples) triplets using a sliding window.
2.  **Define the Model:** Implement a dual-embedding architecture (Context matrix $E$ and Parameter matrix $\theta$) that uses a dot product to measure word similarity.
3.  **Define the Loss Objective:** Utilize the Negative Sampling loss function, which combines the log-sigmoids of positive and negative scores.
4.  **Train the Model:** Iterate through the dataset, using backpropagation to adjust vectors until words with similar meanings cluster together.
5.  **Extract and Save Weights:** Isolate the trained (Matrix $E$) and save it to a file, discarding the temporary classification parameters.
6.  **Load for Inference:** Re-initialize a standalone embedding layer with the saved weights to perform similarity searches or analogy tasks.

### **1. Preparing and Loading the Dataset**

To train a model using **Negative Sampling**, we must structure our data into batches that contrast real word pairings with random noise. This requires a custom **Dataset** class to manage the logic of context-target pairing and a **DataLoader** to handle batching and shuffling. Note that we do not process raw text directly; instead, we pass **numerical indices** mapped from our vocabulary after the tokenization step.

Our custom `Word2VecDataset` class processes a list of sequences and, for every word, generates a training triplet:
1.  **Context:** The index of the current "center" word.
2.  **Positive Target:** The index of a word found within a local sliding window (e.g., $\pm 2$ words).
3.  **Negative Targets:** A set of $k$ random indices from the vocabulary that are *not* the positive target. These are sampled either uniformly or using the empirical **3/4 power frequency distribution** to favor more "informative" negative examples.

#### **Key technical Notes on Implementation of `Word2VecDataset` class**

* **Memory Efficiency & Regularization:**: We generate `negatives` dynamically inside the `__getitem__` method. By doing this, we avoid storing a massive, static dataset on disk. This "on-the-fly" sampling ensures that the model sees different random noise every time it encounters the same positive pair, acting as a powerful form of regularization.

* **Seamless Batching:** The PyTorch `DataLoader` automatically stacks these individual triplets into tensors. For example, with a `batch_size` of $64$ and $k=5$, the negative target tensor becomes $(64, 5)$. This perfectly aligns with the dimensions required for the **Batch Matrix Multiplication** (`torch.bmm`) used in our model's forward pass (see "Defining the Model" section.)

#### **Technical Note about `__getitem__` Method:**

This "magic method" is not fully implemented in the base `torch.utils.data.Dataset` class. It exists only as a placeholder that PyTorch requires us to define ourselves when we create a custom subclass like our `Word2VecDataset` so that the `DataLoader` knows how to fetch a specific data point (like a single Word2Vec triplet) from our data source.

```Python
def __getitem__(self, index):
    raise NotImplementedError

```

Once we defined our `__getitem__` method, we can use square bracket notation on the object to access data point:


```Python
dataset = Word2VecDataset(data, vocab_size)

# This call internally triggers dataset.__getitem__(0)
first_sample = dataset[0]

```

In [ ]:
import torch
from torch.utils.data import Dataset
import random
import numpy as np


class Word2VecDataset(Dataset):
    """A PyTorch Dataset for Skip-gram word2vec with Negative Sampling.

    This class processes sequences of token indices to generate (context, target)
    positive pairs and dynamically samples negative examples to facilitate 
    training via binary logistic regression.

    Attributes:
        vocab_size (int): Total number of unique tokens in the vocabulary.
        k (int): Number of negative samples to generate for each positive pair.
        pairs (list[tuple[int, int]]): A list of all positive (context, target) word pairs.
        heuristic (bool): Whether to use the 3/4 power heuristic for negative sampling.
        unigram_probs (torch.Tensor, optional): The probability distribution used 
            for heuristic negative sampling.
    """

    def __init__(self, data, vocab_size, window_size=2, k=5, heuristic_sampling=True):
        """Initializes the Word2VecDataset with sequences and sampling parameters.

        Args:
            data (list[list[int]]): A list of tokenized sequences (e.g., sentences) 
                represented as integer indices.
            vocab_size (int): Total size of the vocabulary.
            window_size (int, optional): The maximum distance between the context 
                word and the target word. Defaults to 2.
            k (int, optional): The number of negative samples per positive pair. 
                Defaults to 5.
            heuristic_sampling (bool, optional): If True, samples negatives based on 
                the frequency distribution raised to the 0.75 power. Defaults to True.
        """
        self.vocab_size = vocab_size
        self.k = k
        self.pairs = []
        self.heuristic = heuristic_sampling
        
        # 1. Generate positive pairs: Slide a window over each sequence to find neighbors.
        # This captures the local semantic context of each token.
        for sequence in data:
            for i, context_word in enumerate(sequence):
                start = max(0, i - window_size)
                end = min(len(sequence), i + window_size + 1)
                
                for j in range(start, end):
                    if i == j: 
                        continue # A word cannot be its own Target
                    self.pairs.append((context_word, sequence[j]))
        
        # 2. Setup Negative Sampling Distribution.
        if self.heuristic:
            # Count occurrences of each token across the entire corpus.
            word_counts = np.zeros(vocab_size)
            for sequence in data:
                for word_idx in sequence:
                    if word_idx < vocab_size:
                        word_counts[word_idx] += 1
            
            # Apply the 3/4 power heuristic. 
            # This 'smooths' the distribution, making rare tokens slightly more 
            # likely to be sampled as noise than their raw frequency would suggest.
            pow_counts = np.power(word_counts + 1e-10, 0.75)
            self.unigram_probs = torch.tensor(pow_counts / np.sum(pow_counts), dtype=torch.float)
        else:
            self.unigram_probs = None

    def __len__(self):
        """Returns the total number of positive pairs in the dataset.

        Returns:
            int: Total count of (context, target) pairs.
        """
        return len(self.pairs)

    def __getitem__(self, idx):
        """Generates a training triplet: (context, target, negatives).

        This method is called by the DataLoader. Negatives are sampled 'on-the-fly'
        to provide better regularization, ensuring the model sees different noise
        in every epoch.

        Args:
            idx (int): The index of the positive pair to retrieve.

        Returns:
            tuple[torch.Tensor, torch.Tensor, torch.Tensor]: A triplet containing:
                - context (LongTensor): The center word index.
                - target (LongTensor): The positive neighbor index.
                - negatives (LongTensor): A vector of 'k' negative sample indices.
        """
        context, target = self.pairs[idx]
        
        if self.heuristic:
            # Sample k negatives using the pre-computed unigram distribution.
            negatives = torch.multinomial(self.unigram_probs, self.k, replacement=True)
            
            # Strict collision check: Ensure no negative sample is identical to 
            # the true positive target.
            for i in range(self.k):
                while negatives[i] == target:
                    negatives[i] = torch.multinomial(self.unigram_probs, 1)
        else:
            # Uniform random sampling for when heuristic is disabled.
            negatives = []
            while len(negatives) < self.k:
                neg_idx = random.randint(0, self.vocab_size - 1)
                if neg_idx != target:
                    negatives.append(neg_idx)
            negatives = torch.tensor(negatives, dtype=torch.long)
        
        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
            negatives # Already is a Longtensor
        )

#### **Generate Mock Data for Testing `Word2VecDataset`**

To verify our implementation, we define a mock dataset with overlapping "tokens." This simulates a corpus where words appear in multiple contexts, allowing the model to learn complex semantic relationships. We assume the sentences have already been tokenized and converted into numerical indices:

```python
# Each sub-list represents a unique "sentence"
example_data = [
    [10, 20, 30, 40], # "I want a glass"
    [30, 40, 50, 60], # "a glass of orange"
    [60, 70, 80]      # "orange juice please"
]
```

This specific structure is ideal for testing the pipeline because it mimics the core mechanics of the **Word2Vec** algorithm. Specifically, the above structure would satisfy the following:

* **Validates the "Bridge" Effect:** By repeating indices across sequences (e.g., **30, 40,** and **60**), we force the model to learn second-order relationships. Even if token **10** ("I") and token **60** ("orange") never appear in the same sequence, the model connects them through their shared relationship with "glass" (**30**).

* **Tests Sequence Boundary Logic:** The varying sequence lengths (4, 4, and 3) ensure that our `Word2VecDataset` properly handles the end of one sequence without incorrectly pairing the last word of one sentence with the first word of the next.

* **Ensures Sampling Diversity:** With a defined `vocab_size` (e.g., 100), this data provides a clear contrast between **Positive Pairs** (the explicit sequences) and **Negative Samples** (the 90+ unused indices), allowing us to verify that our `3/4` power heuristic is working.

* **Simulates Semantic Clustering:** The logical "flow" of indices (10 $\to$ 80) mimics the way natural language builds meaning. This allows us to perform a quick **Cosine Similarity** check after training to see if neighboring indices (like 60 and 70) have actually clustered together in the embedding space.

In [ ]:
from torch.utils.data import DataLoader


# 1. Mock Data: 3 sequences (e.g., short sentences)
# Each number is a word index from a vocab of size 100
example_data = [
    [10, 20, 30, 40], # "I want a glass"
    [30, 40, 50, 60], # "a glass of orange"
    [60, 70, 80]      # "orange juice please"
]
vocab_size = 100

# 2. Initialize Dataset
# This should generate 26 tuples of (context, positive, negatives)
dataset = Word2VecDataset(example_data, vocab_size=vocab_size, window_size=2, k=5)

# 3. Initialize DataLoader
# shuffle=True is critical for stochastic gradient descent
# This should load 13 batches of datasets
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# 4. Verification
for c, t, n in dataloader:
    print(f"Contexts: {c}")
    print(f"Targets:  {t}")
    print(f"Negatives:\n{n}")
    print(10*"-")
print(len(iter(dataloader)))

### **2. Defining the Model**

In modern deep learning libraries like PyTorch or TensorFlow, defining a model is typically done using an **Embedding Layer** for both the context and the target, followed by a **Dot Product** to compare the embeddings (See [Why dot product?](#why-dot-product) for details).

#### **Proper Initialization of Embedding Layers**

To ensure efficient learning, we utilize an **asymmetric initialization** strategy for the $E$ (input) and $\theta$ (output) matrices. Specifically, $E$ is initialized with small, uniform random values, while $\theta$ is initialized to zero.

* **Breaking Symmetry with Random $E$:** Initializing the input matrix $E$ with **random noise** ensures that every word or token begins at a unique coordinate in the embedding space. This "breaks symmetry," preventing different tokens from receiving identical gradient updates. This allows the model to differentiate between tokens—pulling similar ones together and pushing dissimilar ones apart—from the very first iteration.

* **Establishing a Neutral Baseline with Zero $\theta$:** The matrix $\theta$ functions as a set of binary classifiers. Initializing these weights to zero is a standard practice in logistic regression, as it establishes a "neutral" baseline where $Sigmoid(0) = 0.5$. Since the model starts with $50\%$ uncertainty, the loss function immediately generates a strong gradient. This signal drives $\theta$ toward $1$ (for neighbors) or $0$ (for noise), guided by the distinct features already present in the randomized $E$ matrix.

>**Note:** In high-performance architectures, **Xavier or Kaiming initialization** is often used for both matrices to maintain stable gradient variance across layers. However, the core principle remains the same: the model requires an initial randomness to begin disentangling and mapping complex semantic relationships.

In [ ]:
import torch.nn as nn


class NegativeSamplingModel(nn.Module):
    """A Word2Vec Skip-gram model implemented with Negative Sampling.

    This model implements the dual-embedding architecture where words are 
    represented by two separate vectors: one for when they act as the center 
    'context' word (Matrix E) and one for when they act as a 'target' or 
    'negative' sample (Matrix Theta).

    Attributes:
        vocab_size (int): Total number of unique tokens in the vocabulary.
        embed_size (int): Dimensionality of the embedding space.
        in_embed (nn.Embedding): The input embedding matrix (Matrix E).
        out_embed (nn.Embedding): The output embedding matrix (Matrix Theta).
    """

    def __init__(self, vocab_size, embed_size):
        """Initializes the model with vocabulary size and embedding dimensions.

        Args:
            vocab_size (int): The number of unique tokens.
            embed_size (int): The number of features in each word vector.
        """
        super().__init__() # Activate internal engine of PyTorch
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        
        # Matrix E: Used to represent the 'context' word in the sliding window.
        self.in_embed = nn.Embedding(vocab_size, embed_size)
        
        # Matrix Theta: Used to represent both positive targets and noise samples.
        self.out_embed = nn.Embedding(vocab_size, embed_size)
        
        # Initialization
        initrange = 0.5 / embed_size
        self.in_embed.weight.data.uniform_(-initrange, initrange)
        self.out_embed.weight.data.uniform_(-0, 0)

    def forward(self, input_context, input_target, input_negatives):
        """Calculates similarity scores for positive and negative word pairs.

        This method performs the dot product operations required for the 
        Negative Sampling objective. It compares a single context word against 
        one true target and 'k' negative samples.

        Args:
            input_context (torch.Tensor): LongTensor of shape [batch_size] 
                containing indices of the context words.
            input_target (torch.Tensor): LongTensor of shape [batch_size] 
                containing indices of the true positive words.
            input_negatives (torch.Tensor): LongTensor of shape [batch_size, k] 
                containing indices of the k negative samples per batch item.

        Returns:
            tuple[torch.Tensor, torch.Tensor]: A tuple containing:
                - pos_score (torch.Tensor): Similarity scores for true pairs, 
                  shape [batch_size].
                - neg_score (torch.Tensor): Similarity scores for noise pairs, 
                  shape [batch_size, k].
        """
        # 1. Lookup Embeddings
        # v_c (context): (batch, embed_size)
        v_c = self.in_embed(input_context)
        
        # u_t (positive target): (batch, embed_size)
        u_t = self.out_embed(input_target)
        
        # u_neg (negative samples): (batch, k, embed_size)
        u_neg = self.out_embed(input_negatives)

        # 2. Positive Score: Element-wise multiplication followed by sum 
        # calculates the dot product (v_c · u_t) for each pair in the batch.
        pos_score = torch.sum(v_c * u_t, dim=1) 
        
        # 3. Negative Score: Batch Matrix Multiplication (bmm).
        # We need to compute (v_c · u_neg_i) for i = 1...k.
        # We unsqueeze v_c to (batch, embed_size, 1) to treat it as a column vector.
        # u_neg (batch, k, embed_size) @ v_c (batch, embed_size, 1) = (batch, k, 1)
        neg_score = torch.bmm(u_neg, v_c.unsqueeze(2)).squeeze() 

        return pos_score, neg_score

### **3. Defining the loss objective**

The objective function's goal is to "reward" the model when it correctly identifies a real word pair and "punish" it when it fails to distinguish random noise. It is based on the logic that a good embedding should make real neighbors have a **high** dot product and random words have a **low** (or negative) dot product.

If we let $c$ be the context, $t$ be the positive target, and $\{n_1, \dots, n_k\}$ be the negative samples, the formula being calculated here is:

$$L = -\left[ \log \sigma(\theta_t^T e_c) + \sum_{i=1}^k \log \sigma(-\theta_{n_i}^T e_c) \right]$$

$$J = \frac{1}{m} \Sigma_{i=1}^m L$$

By minimizing this $L$, we are effectively forcing the model to rearrange its embeding layer so that relavent neighbors cluster together and random junk is pushed far away.

* **"Reward" for Correct Pairs:** In loss formula above, the "Reward" for Correct Pairs is:

    $$\text{Reward} = \log \sigma(\theta_t^T e_c)$$

    The goal is here is to maximize the probability that the "Positive" pair (e.g., *orange* and *juice*) is labeled as $1$. If the model is doing well, $\theta_t^T e_c$ is a large positive number. Therefore:

    $$\sigma(\text{large reward}) \approx 1 \to log(1)\approx 0$$

    This is the ideal "maximum" value for log-probability, which pushes the embeddings of $e_{orange}$ and $e_{juice}$ to point in the same direction in the embeding space.

* **The "Punishment" for Random Noise:** In loss formula above, the punishment for Negative Pairs is:

    $$\text{Punishment} = \log \sigma(-\theta_{n_i}^T e_c)$$

    The goal is to maximize the probability that the Negative pairs (e.g., *orange* and *king*) are labeled as $0$. If the model is doing well, $\theta_{n_i}^T e_c$ should be a large negative number and $-\theta_{n_i}^T e_c$ becomes a large positive number. Therefore:

    $$\sigma(\text{large panishment}) \approx 1 \to \log(1) \approx 0$$

    We sum the punishment over $k$ negative examples. We want to accumulate the "correctness" of all $k$ negative samples.

<br>

>**Note:** In deep learning, optimizers (like Adam) are designed to minimize a value. Since we want to maximize log-probability (bringing it closer to 0), we multiply by $-1$ to turn it into a "Loss" that we want to minimize. Also, we take the average over the entire batch so the gradient updates aren't too erratic.

In [ ]:
class NegativeSamplingLoss(nn.Module):
    """Calculates the Noise Contrastive Estimation loss for Word2Vec.

    This loss function implements the negative sampling objective introduced in 
    the original Word2Vec paper. It maximizes the probability of the true 
    neighbor (positive pair) while simultaneously minimizing the probability 
    of 'k' noise samples (negative pairs).

    The mathematical formulation is:
    Loss = -[ log(sigmoid(pos_score)) + sum(log(sigmoid(-neg_scores))) ]
    """

    def __init__(self):
        """Initializes the NegativeSamplingLoss module."""
        super().__init__()

    def forward(self, pos_score, neg_score):
        """Computes the Negative Sampling loss for a batch of samples.

        Args:
            pos_score (torch.Tensor): Predicted similarity scores for true 
                context-target pairs. Shape: [batch_size].
            neg_score (torch.Tensor): Predicted similarity scores for negative 
                samples. Shape: [batch_size, k].

        Returns:
            torch.Tensor: The scalar mean loss value for the batch, 
                suitable for backpropagation.
        """
        # 1. Calculate Positive Loss
        pos_loss = torch.log(torch.sigmoid(pos_score))
        
        # 2. Calculate Negative Loss
        neg_loss = torch.sum(torch.log(torch.sigmoid(-neg_score)), dim=1)
        
        # 3. Combined Average Loss:
        return -torch.mean(pos_loss + neg_loss)

### **4. Training the Model**

During every training step of a Word2Vec model—specifically the **Skip-gram with Negative Sampling** architecture—**both** the Context Embeddings (matrix $E$) and the Target/Parameter Embeddings (matrix $\theta$) are updated. 

When we perform the backward pass (`loss.backward()`), the gradients flow through the dot product calculation and back into both embedding layers:

* **Context Embeddings Update ($E$):** The model adjusts the vector $e_c$ for the **Context word** (e.g., *orange*) to bring it mathematically closer to the real target and further from the noise.
* **Target/Parameter Embeddings Update ($\theta$):** The model adjusts the vector $\theta_t$ for the **Target word** (e.g., *juice*) to make it more receptive to the context word *orange*. Simultaneously, it pushes the vectors $\theta_n$ of the **Negative words** (e.g., *king*, *book*) away from *orange*.

#### **The Mechanics of Gradient Management in PyTorch**

When training deep learning models, how we handle the "mathematical memory" of our gradients is just as critical as the model architecture itself. Understanding the interaction between the optimizer, the backward pass, and the memory buffers is key to stable convergence.

##### **Gradient Accumulation**
In PyTorch, gradients are **accumulated** by default. Mathematically, every time we call `loss.backward()`, PyTorch performs an addition operation:
$$\nabla_{W} L_{total} = \nabla_{W} L_{old} + \nabla_{W} L_{new}$$
Calling `optimizer.zero_grad()` resets this buffer to zero: $\nabla_{W} L = \mathbf{0}$.

If we don't reset the gradients between batches, the model effectively tries to optimize for the current batch while being pulled by the "ghosts" of every previous batch. This leads to massive, exploding gradient values that cause the training to fail.

##### **Why not reset automatically?**
PyTorch’s decision not to reset gradients automatically is a strategic "hardware hack." If our GPU can only fit a batch of **8**, but our model requires a batch of **64** to be stable, we can run 8 consecutive batches of 8, calling `backward()` after each one **without** zeroing the gradients. On the 8th iteration, the buffer contains the sum of all 64 samples. Calling `optimizer.step()` and `zero_grad()` only once at the end allows us to train massive models on consumer-grade hardware by "polling" the data over multiple steps.

##### **Performance Optimization: `set_to_none=True`**
In modern PyTorch, we will often see:
`optimizer.zero_grad(set_to_none=True)`

While mathematically identical to setting gradients to zero, it is computationally faster. Instead of performing an $O(N)$ write operation to fill memory with zeros, it simply deletes the gradient tensor (`None`). PyTorch then allocates a new tensor during the next `backward()` pass, reducing total memory "write" operations.


#### **The `__call__` vs. `forward()` Distinction**

In Python, defining a `__call__` method allows an instance of a class to be treated like a function:

```python
class Greeter:
    def __call__(self, name):
        return f"Hello, {name}!"

say_hi = Greeter()
print(say_hi("Student"))  # Triggers __call__ automatically
```

Every PyTorch model inherits from `nn.Module`, which implements `__call__` to act as a **manager** for our model. Its internal logic looks roughly like this:

```python
# Simplified internal PyTorch source
class Module:
    def __call__(self, *input, **kwargs):
        # 1. Pre-processing: Background "Hooks" 
        # 2. THE CORE CALL: Executes your forward() method
        result = self.forward(*input, **kwargs) 
        # 3. Post-processing: State management and cleanup
        return result
```

When we run `model(context_idx, ...)`, we are executing the `__call__` method, which in turn runs our `forward` logic. While we *could* call `model.forward()` directly, it is **bad practice**. Using the functional call `model()` ensures PyTorch performs essential background tasks:
* **Hooks:** Triggers "Forward Hooks" used for debugging or visualizing internal activations (e.g., Grad-CAM).
* **State Management:** Handles internal logic for distributed training and hardware acceleration.
* **Safety Checks:** Verifies that inputs, weights, and device placements (CPU/GPU) are properly aligned.


#### **Real-time Monitoring with `tqdm`**

To monitor the "heartbeat" of our training, we wrap our `DataLoader` with the `tqdm` library. This provides a visual progress bar with critical real-time feedback, including iterations per second, elapsed time, and Estimated Time of Completion (ETA). 

**Basic `tqdm` Example:**
```python
from tqdm import tqdm
import time

# Wrapping an iterable to create a progress bar
for i in tqdm(range(10), desc='Processing Items'):
    time.sleep(1) # Simulate heavy computation
```

By combining these tools—proper gradient management, functional model calls, and real-time monitoring—we create a training loop that is not only mathematically sound but also efficient and easy to debug.

In [ ]:
import torch.optim as optim
from tqdm import tqdm


# --- Hyperparameters ---
# vocab_size: Total unique tokens (e.g., words or amino acid motifs).
# embed_size: The number of dimensions in the learned semantic space.
# k: The ratio of noise to signal; for each positive pair, we sample k negatives.
vocab_size = 10000
embed_size = 300
k = 5 

# --- Initialization ---
# Initialize the dual-embedding architecture.
model = NegativeSamplingModel(vocab_size, embed_size)

# The Log-Sigmoid objective function.
criterion = NegativeSamplingLoss()

# Adam optimizer handles the adaptive learning rate for every individual 
# parameter (coordinate) in our embedding matrices.
optimizer = optim.Adam(model.parameters(), lr=0.001)

# History containers for visualization and debugging.
batch_loss_history = []
epoch_loss_history = []

# --- Training Execution ---
epochs = 20
for epoch in range(epochs):
    total_epoch_loss = 0
    
    # tqdm provides a visual progress bar for real-time monitoring.
    # Assume 'dataloader' is a PyTorch DataLoader yielding triplets.
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    
    # Set model to training mode. This ensures that internal state (like 
    # gradients) is tracked and that dropout (if any) is active.
    model.train() 
    
    for context_idx, target_idx, negative_indices in progress_bar:
        # 1. Zero Gradients: PyTorch accumulates gradients by default. 
        # We must clear the 'workbench' before starting a new calculation.
        optimizer.zero_grad(set_to_none=True)
        
        # 2. Forward Pass: Computes the similarity scores for the batch.
        # Calling model() triggers the .forward() method internally.
        pos_scores, neg_scores = model(context_idx, target_idx, negative_indices)
        
        # 3. Compute Loss: Apply the Negative Sampling (Noise Contrastive) objective.
        loss = criterion(pos_scores, neg_scores)
        
        # 4. Backward Pass: Calculate the gradient of the loss with respect 
        # to every weight in E and Theta using the chain rule.
        loss.backward()
        
        # 5. Optimizer Step: Update the weights based on the gradients.
        # This is where the vectors actually "move" in the embedding space.
        optimizer.step()
        
        # 6. Logging: Extract the scalar value from the loss tensor.
        current_loss = loss.item()
        batch_loss_history.append(current_loss)
        total_epoch_loss += current_loss

        # Update the progress bar with the most recent batch loss.
        progress_bar.set_postfix({'loss': f"{current_loss:.4f}"})
        
    # Calculate and store the average loss for the entire epoch.
    avg_epoch_loss = total_epoch_loss / len(dataloader)
    epoch_loss_history.append(avg_epoch_loss)
    print(f"--- Epoch {epoch+1} Complete. Avg Loss: {avg_epoch_loss:.4f} ---")

Let's visualize how how the loss changes in each training epoch:

In [ ]:
import matplotlib.pyplot as plt


# 1. Setup the figure and axes
fig, axes = plt.subplots(1, 2, figsize=(10, 4), squeeze=False)
ax1, ax2 = axes.flatten()

# 2. Plot Batch-level Loss
# This shows the "noisy" loss calculated at every individual step.
# High volatility here is normal but should generally trend downward.
ax1.plot(batch_loss_history)
ax1.set_title("Training Loss (Per Batch)")
ax1.set_xlabel("Batch Iteration")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3) # Added for better readability

# 3. Plot Epoch-level Loss
# This provides a "smoothed" view of the model's overall progress.
ax2.plot(epoch_loss_history, '-o', color='orange')
ax2.set_title("Average Loss (Per Epoch)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Mean Loss")
ax2.grid(True, alpha=0.3)

# Adjust layout to prevent titles and labels from overlapping
plt.tight_layout()
plt.show()

### **5. Extracting and Saving Weights**

Saving our trained embeddings is the critical final step of the pipeline. Once training is complete, the `out_embed` (the $\theta$ parameters) has fulfilled its role as a temporary binary classifier. The `in_embed` (the $E$ matrix) is the "gold" representation we intend to preserve for downstream applications.

#### **Embedding Selection:**
Matrix $E$ is what is traditionally referred to as "The Word Embeddings." However, some researchers prefer to use the **average** of the two matrices ($(E + \theta)/2$) to produce a final representation, which can occasionally be more robust. In most standard pipelines, Matrix $\theta$ is discarded after training because its primary function was to serve as the "classifier weights" for the negative sampling task. We will use `torch.save` to export the internal `state_dict` of the embedding layer to a `.pt` or `.pth` file for persistent storage.

#### **Why Export Only the `in_embed` Weights?**
By saving only the `in_embed` matrix, we are effectively exporting the "semantic map" discovered by the model while discarding the auxiliary parameters used during training. This approach offers two major advantages:

* **Reduced Memory Footprint:** The resulting file is significantly smaller because it excludes the `out_embed` parameters ($\theta$) and the internal states of the optimizer (like Adam’s momentum buffers).
* **Plug-and-Play Utility:** In a production environment, these weights act as a high-quality "head start" for more complex architectures, such as **Transformers**, which benefit from initialized "meanings" rather than random noise.

In [ ]:
# Path to save the file
SAVE_PATH = "embeddings_v1.pt"

# Extract only the 'in_embed' weights (the E matrix)
# This is a tensor of shape [vocab_size, embed_size]
embedding_weights = model.in_embed.state_dict()

# Save to disk
torch.save(embedding_weights, SAVE_PATH)
print(f"Successfully saved embeddings to {SAVE_PATH}")

### **6. Loading the Embeddings for Inference**

To perform downstream tasks like similarity search or analogy reasoning, we no longer require the full `NegativeSamplingModel`. Instead, we can use a standard `nn.Embedding` layer as a container for our pre-trained weights. Once these weights are loaded, we can identify semantically related tokens by calculating the **Cosine Similarity** between their respective vectors.

#### **Turning on Evaluation Mode**

Calling `final_embeddings.eval()` is a critical step that transitions a PyTorch model from **Training Mode** to **Evaluation Mode**. Briefly, calling `eval()` method:

* Flips a global boolean flag (`self.training`) within the model. This tells specific layers—like **Dropout** and **Batch Normalization**—to stop their training-specific behaviors (like randomly dropping neurons or updating running means) and use their "learned" stable configurations instead.
* Ensures that the model's output is deterministic. During inference, we want the same input to produce the exact same output every time, which wouldn't happen if Dropout remained active.
* Is an industry-standard safeguard that ensures the model is prepared for inference even if our model only contains simple Embedding layers.

#### **Crucial Distinction: `.eval()` vs. `torch.no_grad()`**
While `.eval()` changes the behavior of the layers, it does **not** stop the mathematical calculation of gradients. To speed up inference and save memory, we must combine it with `torch.no_grad()` context manager:

```python
model.eval()           # Set behavior (Disable Dropout)
with torch.no_grad():  # Set memory (Disable Gradient math)
    # Inference code here
```

In [ ]:
def get_most_similar(word_idx, embedding_layer, top_k=5):
    """Identifies the nearest neighbors of a word in the embedding space.

    This function performs a similarity search across the entire vocabulary 
    using Cosine Similarity, which measures the angular distance between vectors.

    Args:
        word_idx (int): The vocabulary index of the query word.
        embedding_layer (nn.Embedding): The trained PyTorch embedding layer.
        top_k (int, optional): Number of similar words to return. Defaults to 5.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: A tuple containing:
            - indices: The vocabulary IDs of the top_k most similar words.
            - values: The corresponding cosine similarity scores (range -1 to 1).
    """
    # 1. Retrieve the vector for the query word. 
    # Shape becomes [1, embed_size].
    target_vec = embedding_layer(torch.tensor([word_idx])) 
    
    # 2. Access the full weight matrix (Matrix E).
    # Shape is [vocab_size, embed_size].
    all_vecs = embedding_layer.weight 
    
    # 3. Calculate Cosine Similarity.
    # We compare the target_vec against all_vecs simultaneously.
    # Cosine Similarity = (A · B) / (||A|| * ||B||)
    cos = nn.CosineSimilarity(dim=1)
    similarities = cos(target_vec, all_vecs)
    
    # 4. Extract top results.
    # We ask for k+1 because the most similar word is always the word itself 
    # (similarity = 1.0). We then slice [1:] to skip the self-match.
    values, indices = torch.topk(similarities, top_k + 1)
    
    return indices[1:], values[1:]

# --- Execution Block ---

# Configuration must match the training setup
vocab_size = 10000 
embed_size = 300

# Re-initialize a standalone layer to hold the trained weights
final_embeddings = nn.Embedding(vocab_size, embed_size)

# Load the saved state: weights_only=True is a security best practice to 
# prevent the execution of malicious pickled code.
final_embeddings.load_state_dict(torch.load("embeddings_v1.pt", weights_only=True))

# 1. .eval() sets the behavior for inference (disables dropout, etc.)
final_embeddings.eval()

# 2. torch.no_grad() disables the computational graph, saving memory and CPU/GPU.
with torch.no_grad():
    # Example: Query the semantic neighbors for index 60 (e.g., 'orange')
    similar_indices, scores = get_most_similar(60, final_embeddings)
    
    print(f"Indices of words similar to 'orange': {similar_indices.tolist()}")
    print(f"Scores of words similar to 'orange': {scores.tolist()}")

#### **Note on Model Performance**
In the example above, we notice that the semantic neighbors for our query word (e.g., *orange*) are not yet perfectly accurate. This is expected behavior. Achieving "human-like" semantic relationships usually requires training on massive corpora (millions of sentences) and more complex architectures.

The primary goal of this exercise was not to produce a production-ready language model, but rather to demonstrate the **standard end-to-end training workflow**: from data preparation and gradient management to weights extraction and similarity search.

Here is why a simple model might struggle initially:

* **Data Sparsity:** If the word "orange" only appears five times in our dataset, the model hasn't seen enough varied contexts to decide if it’s a fruit, a color, or a brand.
* **The "Alignment" Problem:** Early in training, the $E$ (Context) and $\theta$ (Target) matrices might not be perfectly aligned. Often, the Context matrix ($E$) contains more "meaning," while the Target matrix ($\theta$) is more biased toward word frequency.
* **Embed Size vs. Vocab Size:** If our `embed_size` is too small for a huge vocabulary, the model suffers from "bottlenecking"—it doesn't have enough "room" to move words away from each other.

---

### **Why Use Dot Product Instead of Cosine Similarity in Training?**

While Cosine Similarity is the gold standard for finding analogies ($Man:Woman :: King:Queen$) after a model is trained, we almost exclusively use the **Dot Product** ($\theta^T e$) during the actual Negative Sampling training phase. 

The decision to avoid Cosine Similarity during training comes down to two factors: **computational efficiency** and how the model encodes **word importance**.

##### **1. The Computational Bottleneck**
The most practical reason is speed. A **Dot Product** is a simple operation consisting only of multiplication and addition. In contrast, **Cosine Similarity** requires calculating the dot product *plus* the $L_2$ norm (the square root of the sum of squares) for every vector in the operation. When training on a corpus of billions of words, adding square root and division operations to every single weight update significantly slows down convergence.

##### **2. The Significance of Vector Magnitude (Norm)**
In a Dot Product, the result is determined by both the **angle** and the **magnitude (length)** of the vectors:

$$u \cdot v = \|u\| \|v\| \cos(\phi)$$

By using Cosine Similarity, we would be normalizing all vectors to a length of $1$, effectively discarding their magnitude. In Word2Vec, however, the length of a vector carries vital information:

* **Frequent Tokens:** Words that appear often (like "the" or "of") receive more frequent gradient updates, leading to larger norms.
* **Rare Tokens:** Words seen only a few times remain closer to the origin, resulting in **smaller norms**.

By utilizing the Dot Product, the model uses vector length to represent "confidence" or "frequency," which allows the objective function to converge more naturally.